# Suite QRY — Queries

A rule is a query: `select(name, rule, variables)` streams a store's objects of a schema (its extent: what its
singletons reach) for which the rule holds. `Queries.Scan(store)` makes any store queryable in memory.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Queries as Q } from "@mbse/patterns";
import { Bindings as Bd, Errors, Schemas as S } from "@mbse/schemas/Framework";
import { Contact, assert, book, listed, native, protocol_problems, raises, same } from "./support.js";

const self = E.variable("this");
const store: any = book();
const ann = store.Contact().name("ann").age(30n).create();
const bob = store.Contact().name("bob").create();
const cy = store.Contact().name("cy").age(9n).create();
listed(store, ann, bob, cy);
const loose = store.Contact().name("dee").age(50n).create(); // never listed: not the store's data
const query = new Q.Scan(store);

## QRY-01 · A query streams the extent's objects for which its rule holds

In [ ]:
{
  const adults = query.select("Contact", self.age.ge(18n));
  assert(adults[Symbol.iterator]() === adults && adults.next().value === ann); // an iterator, read as it is asked
  assert([...adults].length === 0); // the rest: bob's age is unknown, cy is a child, dee is not in the extent
  assert(same(query.select("Contact", self.age.ge(18n), null, true), [ann, bob]));
  assert(same(query.select("Contact", self.has("name")), [ann, bob, cy]));
}

## QRY-02 · Variables are bound once, and objects too

In [ ]:
{
  assert(same(query.select("Contact", self.age.ge(E.variable("min")), { min: 21n }), [ann]));
  assert(same(query.select("Contact", self.age.lt(E.variable("max")).and_(self.ne(E.variable("not_"))),
    { max: 40n, not_: cy }), [ann])); // an object has no literal: it stays a variable, bound per object
  assert(same(query.select("Contact", E.variable("flag").or_(self.age.ge(99n)), { flag: true }), [ann, bob, cy]));
}

## QRY-03 · The extent is read when the first match is asked for

In [ ]:
{
  const pending = query.select("Contact", self.age.ge(40n));
  listed(store, loose); // listed after the query was made, before it was read
  assert(same(pending, [loose]));
}

## QRY-04 · A query that cannot be right is refused when it is made

In [ ]:
{
  raises(Errors.AttributeError, () => query.select("Nope", self.has("name")), "no schema registered as 'Nope'");
  raises(TypeError, () => query.select("Listed", self.has("name")), "'Listed' is a relation");
  raises(Errors.ValueError, () => query.select("Contact", self.has("name"), { this: ann }),
    "'this' is bound to each object; it is not a variable");
  raises(Errors.ValueError, () => query.select("Contact", self.age.ge(E.variable("n"))), "the rule cannot be a query: ");
  raises(Errors.ValueError, () => query.select("Contact", E.operation("pow", self.age, 2n).ge(1n)), "'pow' is not a core operation");
  const numbers = query.select("Contact", self.age); // a rule must give a bool, found as objects are read
  raises(TypeError, () => [...numbers], "a query's rule must give a bool");
}

## QRY-05 · Scan is a store, and select asks a queryable store or scans another

In [ ]:
{
  assert(protocol_problems(query).length === 0 && same(protocol_problems(store), ["select"]));
  assert(query.schema("Contact") === Contact && query.registered("Listed") === store.registered("Listed")
    && query.name_of(Contact) === "Contact");
  assert(same(query.names(), store.names()) && same(query.extent("Contact"), store.extent("Contact")));
  assert(query.singleton("book.Directory") === store.singleton("book.Directory"));
  assert(query.member(ann, "name") === "ann" && query.builder("Contact").name("e").create().name === "e");
  assert(query.builder("Contact", ann).update() === ann && query.store === store);
  const scanned = Q.select(store, "Contact", self.age.ge(18n)); // a plain store is scanned; the update above reordered its entries
  assert(same([...scanned].map((c: any) => c.name).sort(), ["ann", "dee"]));

  /** A store that answers queries itself. */
  class Native {
    select(name: string, rule: unknown, variables: unknown = null, unknown = false) {
      return [["asked", name, variables, unknown]][Symbol.iterator]();
    }
  }

  const [asked] = [...Q.select(new Native() as any, "X", self.has("a"), { v: 1n }, true)] as any[];
  assert(asked[0] === "asked" && asked[1] === "X" && asked[2].v === 1n && asked[3] === true);
  const Item = new S.OfObject.Builder().ref().singleton("the.Item").properties(native("name")).create();

  class Thing {
    constructor(public name: string | null) {}

    identity(): unknown {
      return this;
    }

    schema_name(): string {
      return "Item";
    }

    owner(): null {
      return null;
    }

    accept(visitor: any): void {
      Bd.accept(binding, this, visitor);
    }
  }

  const binding: Bd.Binding = new Bd.Binding(Item, (t: Thing) => new Bd.State(new Map([["name", t.name]])),
    (s) => new Thing((s.values.get("name") as string | undefined) ?? null));
  const bound = new Bd.OfStore(new Map([["Item", [Item, (instance?: Thing) => new Bd.Builder(binding, instance)] as const]]));
  assert(same(Q.select(bound, "Item", self.has("name").not_()), [bound.singleton("the.Item")])); // bound classes too
}